# PlantCLEF 2015 LeafScan Training Bundle

Utility notebook for building `PlantCLEF2015_leafscan_only.tar.gz` from the official full PlantCLEF 2015 training package and saving the compact LeafScan archive to Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Clone Or Update Project

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


## 3. Download And Extract Full Training Package

The full archive is downloaded to local Colab disk, not directly to Google Drive. This avoids Drive FUSE write failures on large files.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

RAW_ARCHIVE=/content/PlantCLEF2015TrainingData.tar.gz
RAW_DIR=/content/plantclef2015_training_raw
URL=https://lab.plantnet.org/LifeCLEF/PlantCLEF2015/TrainingPackage/PlantCLEF2015TrainingData.tar.gz

df -h /content
if [ -s "$RAW_ARCHIVE" ] && tar -tzf "$RAW_ARCHIVE" >/dev/null 2>&1; then
  echo "using existing validated local training archive: $RAW_ARCHIVE"
else
  echo "downloading full PlantCLEF 2015 training archive to local Colab disk"
  wget -c --tries=20 --timeout=120 --read-timeout=120 "$URL" -O "$RAW_ARCHIVE"
fi
tar -tzf "$RAW_ARCHIVE" >/dev/null

rm -rf "$RAW_DIR"
mkdir -p "$RAW_DIR"
tar -xzf "$RAW_ARCHIVE" -C "$RAW_DIR"
echo "files extracted:"
find "$RAW_DIR" -type f | wc -l
find "$RAW_DIR" -type f -iname '*.xml' | wc -l
find "$RAW_DIR" -type f \( -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' \) | wc -l


## 4. Build Local LeafScan Archive

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

RAW_DIR=/content/plantclef2015_training_raw
LOCAL_BUNDLE=/content/PlantCLEF2015_leafscan_only.tar.gz
WORK_DIR=/content/plantclef2015_leafscan_bundle_work

rm -f "$LOCAL_BUNDLE"
python scripts/build_plantclef_content_bundle.py \
  --source-root "$RAW_DIR" \
  --image-root "$RAW_DIR" \
  --output "$LOCAL_BUNDLE" \
  --content LeafScan \
  --work-dir "$WORK_DIR"

ls -lh "$LOCAL_BUNDLE"
tar -tzf "$LOCAL_BUNDLE" | sed -n '1,20p'


## 5. Validate Local Bundle

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

LOCAL_BUNDLE=/content/PlantCLEF2015_leafscan_only.tar.gz
CHECK_DIR=/content/plantclef2015_leafscan_check
rm -rf "$CHECK_DIR"
mkdir -p "$CHECK_DIR"
tar -xzf "$LOCAL_BUNDLE" -C "$CHECK_DIR" leafscan/metadata.csv
python - <<'PY_CHECK'
import csv
from collections import Counter
from pathlib import Path
metadata = Path('/content/plantclef2015_leafscan_check/leafscan/metadata.csv')
with metadata.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
content = Counter(row.get('content', '') for row in rows)
genera = len({row['genus'] for row in rows})
species = len({row['species'] for row in rows})
print('rows:', len(rows))
print('content:', content)
print('genera:', genera)
print('species:', species)
if len(rows) != 12605 or content != Counter({'LeafScan': 12605}):
    raise RuntimeError(f'Expected 12605 LeafScan rows, got {len(rows)} / {content}')
PY_CHECK
sha256sum "$LOCAL_BUNDLE"


## 6. Copy LeafScan Archive To Google Drive

Only the compact LeafScan archive is copied to Drive. The full raw package stays on temporary Colab disk.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

LOCAL_BUNDLE=/content/PlantCLEF2015_leafscan_only.tar.gz
DRIVE_BUNDLE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz

test -f "$LOCAL_BUNDLE"
mkdir -p /content/drive/MyDrive
rm -f "$DRIVE_BUNDLE"
cp "$LOCAL_BUNDLE" "$DRIVE_BUNDLE"
sync
ls -lh "$DRIVE_BUNDLE"
tar -tzf "$DRIVE_BUNDLE" >/dev/null
echo "Drive archive validated: $DRIVE_BUNDLE"


## 7. Optional Cleanup

In [ ]:
%%bash
set -euo pipefail

rm -rf /content/plantclef2015_training_raw \
       /content/plantclef2015_leafscan_bundle_work \
       /content/plantclef2015_leafscan_check
rm -f /content/PlantCLEF2015TrainingData.tar.gz \
      /content/PlantCLEF2015_leafscan_only.tar.gz
df -h /content
